In [ ]:
%pip install folium

import pandas as pd
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
import branca.colormap as cm
import re
import json
import geodatasets

import ipywidgets as widgets
from IPython.display import HTML, display

import folium
import matplotlib
import mapclassify

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\annas\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [62]:
yrs = list(range(15, 26))
df_name = ["labor_" + str(x) for x in yrs]

datasets = {}
for k, v in zip(df_name, yrs):
    a = pd.read_excel(f"laucnty{str(v)}.xlsx", skiprows=1)
    datasets[k] = a

#stack data
data_frame_labor15_25 = pd.concat(datasets.values(), ignore_index=True)

#remove PR, VI + others
territories = [60, 66, 69, 72, 74, 78]
data_frame_labor15_25 = data_frame_labor15_25[~data_frame_labor15_25['State FIPS Code'].isin(territories)]

#add county polygon data, rename columns
counties = gpd.read_file("counties.geojson")
counties[['STATEFP', 'COUNTYFP' ]] = counties[['STATEFP', 'COUNTYFP' ]].astype(float)
counties = counties.rename(columns={'STATEFP': 'State FIPS Code', 'COUNTYFP': 'County FIPS Code'}) 

#CT has new "planning regions," replace CT counties with planning regions
json_file = ("CT_Planning_Regions.geojson")

with open(json_file) as f:
    ct_counties = json.load(f)

name = []
FIPS_code = []
polygon = []

for i in range(0, 9):
    name.append(ct_counties['features'][i]['properties']['PlanningRegion']) #county name
    FIPS_code.append(ct_counties['features'][i]['properties']['PlanningRegionFIPS']) #county FIPS code
    polygon.append(ct_counties['features'][i]['geometry']['coordinates']) #polygon

ct_counties = pd.DataFrame({"NAME":name, "County FIPS Code":FIPS_code, "State FIPS Code": 9.0, "geometry":polygon})

#function to find the polygon file within nested list
def polygon_nested_lists(a):
    count = 0
    og_a = a

    while a:
        if isinstance(a, list):
            count += 1
            a = a[0]
            continue
        else: 
            for i in range(0, count-2):
                og_a = og_a[0]
            return Polygon(og_a)

ct_counties['geometry'] = ct_counties['geometry'].apply(lambda x: polygon_nested_lists(x))
ct_counties['County FIPS Code'] = ct_counties['County FIPS Code'].astype(float)

#remove previous counties and merge
counties = counties[counties['State FIPS Code'] != 9.0] 
counties = pd.concat([counties, ct_counties], ignore_index=True)

#merge unemployment data with polygon data
data_frame_labor15_25_clean = pd.merge(data_frame_labor15_25, counties[['State FIPS Code', 'County FIPS Code', 'geometry']], on=['State FIPS Code', 'County FIPS Code'], how='inner')


In [63]:
###MAP OF 2025 UNEMPLOYMENT DATA BY STATE AND COUNTY

#add state abbreviation data
data_frame_labor15_25_clean['ABR'] = data_frame_labor15_25_clean.apply(lambda x: x['County Name/State Abbreviation'][-2:] if x['State FIPS Code'] != 11.0 else "DC", axis = 1)
state_names = pd.read_excel("state_abr.xlsx")
data_frame_labor15_25_clean = pd.merge(data_frame_labor15_25_clean, state_names, on='ABR', how='left')

data_frame_labor15_25_clean_2025 = data_frame_labor15_25_clean[data_frame_labor15_25_clean['Year'] == 2025.0]

#create base map
map = folium.Map(location =[38.386682, -97.955583],  
        tiles = 'OpenStreetMap',
        attr='&copy; <a href="https://openstreetmap.org">OpenStreetMap</a> contributors &copy; <a href="https://carto.com">CARTO</a>',
        zoom_start=5)

gdf = gpd.GeoDataFrame(data_frame_labor15_25_clean_2025, geometry="geometry", crs="EPSG:4326")

gdf = gdf[['State', 'County Name/State Abbreviation', 'geometry', "Unemployment Rate (%)"]]

gdf['County'] = gdf['County Name/State Abbreviation'].apply(lambda x: x[:-4])
gdf['Unemployment'] = gdf['Unemployment Rate (%)'].apply(lambda x: str(x) + "%")

min_val = data_frame_labor15_25_clean_2025['Unemployment Rate (%)'].min()
max_val = data_frame_labor15_25_clean_2025['Unemployment Rate (%)'].max()

colormap = cm.linear.Blues_09.scale(min_val, max_val)
colormap.caption = 'Unemployment Rate'

gdf['color_map_hex'] = gdf['Unemployment Rate (%)'].apply(lambda x: colormap(x))

state_group = {}
state_list = gdf['State'].unique().tolist()
state_list.insert(0, 'All')

county_group = {}
county_list = gdf['County Name/State Abbreviation'].unique().tolist()
county_list.insert(0, 'All')

for c in state_list:
    df = gdf.query('State == @c')
    state_group[c] = df['County Name/State Abbreviation'].tolist()

out = widgets.Output(layout={'border': '1px solid black'})

s = widgets.Dropdown(
    options=state_list,
    value=state_list[0],
    description='State:',
    disabled=False,
)

c = widgets.Dropdown(
    description='County:',
    disabled=False,
)

def state_update(x):
    county_list = state_group[x]
    c.options = county_list


# Define a function to filter the dataframe based off value in dropdown menu
def filter_dataframe(state_name, county_name=None):
  if county_name is None:
    return gdf[gdf['State'] == state_name]
  else:
    return gdf[(gdf['State'] == state_name) & (gdf['County Name/State Abbreviation'] == county_name)]

def dropdown_eventhandler1(change):
    state_update(s.value)
    out.clear_output()
    df = filter_dataframe(s.value, c.value)

    with out: 
        display(df.explore(tooltip=['County', 'Unemployment'],
                style_kwds = dict(
                style_function=lambda x: {
                    "fillColor": (x["properties"]['color_map_hex']),
                    "color": "black",
                    "weight": 0.25,
                    "fillOpacity": 1,
                })))

def dropdown_eventhandler2(change):
    out.clear_output()
    df = filter_dataframe(s.value, c.value)
    with out: 
        display(df.explore(tooltip=['County', 'Unemployment'],
                           style_kwds = dict(
                style_function=lambda x: {
                    "fillColor": (x["properties"]['color_map_hex']),
                    "color": "black",
                    "weight": 0.2,
                    "fillOpacity": 1,
                })))


s.observe(dropdown_eventhandler1, names='value')
c.observe(dropdown_eventhandler2, names='value')

display(s, c) #this displays the actual map

with out:
    display(gdf.explore(
       location =[38.386682, -97.955583],
        zoom_start=4.25,
        tooltip=['County', 'Unemployment'],                         
            style_kwds = dict(
            style_function=lambda x: {
                "fillColor": (x["properties"]['color_map_hex']),
                "color": "black",
                "weight": 0.2,
                "fillOpacity": 1
            })))


out

Dropdown(description='State:', options=('All', 'Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colo…

Dropdown(description='County:', options=(), value=None)

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

In [64]:
gdf = gpd.GeoDataFrame(data_frame_labor15_25_clean, geometry="geometry", crs="EPSG:4326")

gdf = gdf[['State', 'Year', 'County Name/State Abbreviation', 'geometry', "Unemployment Rate (%)"]]

gdf['County'] = gdf['County Name/State Abbreviation'].apply(lambda x: x[:-4])

min_val = data_frame_labor15_25_clean_2025['Unemployment Rate (%)'].min()
max_val = data_frame_labor15_25_clean_2025['Unemployment Rate (%)'].max()

colormap = cm.linear.Blues_09.scale(min_val, max_val)
colormap.caption = 'Unemployment Rate'

def tooltip_text(x):
    rate = x["Unemployment Rate (%)"]
    county = x['County']
    text = f"<b>Unemployment:</b> {rate}% <br> <b>County:</b> {county}"
    return text

slider = widgets.IntSlider(min=2015, max=2025, step=1, description="Year", orientation='horizontal')
out = widgets.Output()


map = folium.Map(location =[38.386682, -97.955583],  
        tiles = 'OpenStreetMap',
        attr='&copy; <a href="https://openstreetmap.org">OpenStreetMap</a> contributors &copy; <a href="https://carto.com">CARTO</a>',
        zoom_start=4)


# Apply a fixed style to all features
def year_map(x):
    folium.GeoJson(
        data = gdf[gdf['Year'] == x],
        style_function=lambda x: {
        "fillColor": colormap(x['properties']['Unemployment Rate (%)']),
        "color": "black",
        "weight": .5,
        "fillOpacity": 0.5,
        #"tooltip": "xx"
        }).add_to(map)

    display(map)

def on_value_change(change):
    with out:
        out.clear_output()
        year_map(change['new'])

# Watch for changes on the 'value' trait
slider.observe(on_value_change, names='value')

display(slider, out)
year_map(2015)


##add colormap
#add tooltip
#add title

IntSlider(value=2015, description='Year', max=2025, min=2015)

Output()